# Feature Engineering

This notebook creates meaningful product features that improve recommendation quality and provide additional business insights.

Features created:

- Discount Percentage
- Price Bucket
- Rating Score
- Category Popularity
- Brand Popularity
- Product Popularity Score

Finally, all engineered features are saved as **feature_store.csv**.

In [1]:
import pandas as pd
import numpy as np

In [2]:
products_df = pd.read_csv(
    "../data/processed/cleaned_electronics_products.csv"
)

products_df.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price
0,0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,4.0,965,"₹10,999","₹18,999"
1,1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,4.3,"113,956","₹18,999","₹19,999"
2,2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51UhwaQXCp...,https://www.amazon.in/Oneplus-Bluetooth-Wirele...,4.2,"90,304","₹1,999","₹2,299"
3,3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81I3w4J6yj...,https://www.amazon.in/Samsung-Mystique-Storage...,4.1,"24,863","₹15,999","₹24,999"
4,4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71V--WZVUI...,https://www.amazon.in/OnePlus-Nord-Black-128GB...,4.3,"113,956","₹18,999","₹19,999"


In [3]:
print(products_df.shape)

products_df.info()

(9600, 10)
<class 'pandas.DataFrame'>
RangeIndex: 9600 entries, 0 to 9599
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Unnamed: 0      9600 non-null   int64
 1   name            9600 non-null   str  
 2   main_category   9600 non-null   str  
 3   sub_category    9600 non-null   str  
 4   image           9600 non-null   str  
 5   link            9600 non-null   str  
 6   ratings         9505 non-null   str  
 7   no_of_ratings   9505 non-null   str  
 8   discount_price  9116 non-null   str  
 9   actual_price    9530 non-null   str  
dtypes: int64(1), str(9)
memory usage: 4.0 MB


In [4]:
print(products_df.columns)

Index(['Unnamed: 0', 'name', 'main_category', 'sub_category', 'image', 'link',
       'ratings', 'no_of_ratings', 'discount_price', 'actual_price'],
      dtype='str')


In [5]:
# Clean Ratings
products_df["ratings"] = pd.to_numeric(
    products_df["ratings"],
    errors="coerce"
)

# Clean Number of Ratings
products_df["no_of_ratings"] = (
    products_df["no_of_ratings"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

products_df["no_of_ratings"] = pd.to_numeric(
    products_df["no_of_ratings"],
    errors="coerce"
)

# Clean Discount Price
products_df["discount_price"] = (
    products_df["discount_price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
)

products_df["discount_price"] = pd.to_numeric(
    products_df["discount_price"],
    errors="coerce"
)

# Clean Actual Price
products_df["actual_price"] = (
    products_df["actual_price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
)

products_df["actual_price"] = pd.to_numeric(
    products_df["actual_price"],
    errors="coerce"
)

products_df[
    [
        "ratings",
        "no_of_ratings",
        "discount_price",
        "actual_price"
    ]
].head()

,ratings,no_of_ratings,discount_price,actual_price
0,4.0,965.0,10999.0,18999.0
1,4.3,113956.0,18999.0,19999.0
2,4.2,90304.0,1999.0,2299.0
3,4.1,24863.0,15999.0,24999.0
4,4.3,113956.0,18999.0,19999.0


In [6]:
products_df["discount_percentage"] = (
    (
        products_df["actual_price"] -
        products_df["discount_price"]
    )
    /
    products_df["actual_price"]
) * 100

products_df["discount_percentage"] = (
    products_df["discount_percentage"]
    .fillna(0)
)

products_df[
    [
        "actual_price",
        "discount_price",
        "discount_percentage"
    ]
].head()

,actual_price,discount_price,discount_percentage
0,18999.0,10999.0,42.107479
1,19999.0,18999.0,5.000250
2,2299.0,1999.0,13.049152
3,24999.0,15999.0,36.001440
4,19999.0,18999.0,5.000250


In [7]:
products_df["price_bucket"] = pd.cut(

    products_df["discount_price"],

    bins=[
        0,
        500,
        1000,
        3000,
        10000,
        np.inf
    ],

    labels=[
        "Budget",
        "Low",
        "Medium",
        "Premium",
        "Luxury"
    ]
)

products_df[
    [
        "discount_price",
        "price_bucket"
    ]
].head()

,discount_price,price_bucket
0,10999.0,Luxury
1,18999.0,Luxury
2,1999.0,Medium
3,15999.0,Luxury
4,18999.0,Luxury


In [8]:
category_popularity = (
    products_df["main_category"]
    .value_counts()
)

products_df["category_popularity"] = (
    products_df["main_category"]
    .map(category_popularity)
)

products_df[
    [
        "main_category",
        "category_popularity"
    ]
].head()

,main_category,category_popularity
0,"tv, audio & cameras",9600
1,"tv, audio & cameras",9600
2,"tv, audio & cameras",9600
3,"tv, audio & cameras",9600
4,"tv, audio & cameras",9600


In [9]:
subcategory_popularity = (
    products_df["sub_category"]
    .value_counts()
)

products_df["subcategory_popularity"] = (
    products_df["sub_category"]
    .map(subcategory_popularity)
)

products_df[
    [
        "sub_category",
        "subcategory_popularity"
    ]
].head()

,sub_category,subcategory_popularity
0,All Electronics,9600
1,All Electronics,9600
2,All Electronics,9600
3,All Electronics,9600
4,All Electronics,9600


In [10]:
products_df["popularity_score"] = (
    products_df["ratings"].fillna(0)
    *
    np.log1p(
        products_df["no_of_ratings"].fillna(0)
    )
)

products_df[
    [
        "name",
        "ratings",
        "no_of_ratings",
        "popularity_score"
    ]
].head()

,name,ratings,no_of_ratings,popularity_score
0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...",4.0,965.0,27.492655
1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...",4.3,113956.0,50.067379
2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,4.2,90304.0,47.925982
3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...",4.1,24863.0,41.496823
4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...",4.3,113956.0,50.067379


In [11]:
products_df.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price,discount_percentage,price_bucket,category_popularity,subcategory_popularity,popularity_score
0,0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,4.0,965.0,10999.0,18999.0,42.107479,Luxury,9600,9600,27.492655
1,1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,4.3,113956.0,18999.0,19999.0,5.000250,Luxury,9600,9600,50.067379
2,2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51UhwaQXCp...,https://www.amazon.in/Oneplus-Bluetooth-Wirele...,4.2,90304.0,1999.0,2299.0,13.049152,Medium,9600,9600,47.925982
3,3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81I3w4J6yj...,https://www.amazon.in/Samsung-Mystique-Storage...,4.1,24863.0,15999.0,24999.0,36.001440,Luxury,9600,9600,41.496823
4,4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71V--WZVUI...,https://www.amazon.in/OnePlus-Nord-Black-128GB...,4.3,113956.0,18999.0,19999.0,5.000250,Luxury,9600,9600,50.067379


In [12]:
products_df.to_csv(
    "../data/processed/feature_store.csv",
    index=False
)

print("✅ Feature Store Saved Successfully!")

✅ Feature Store Saved Successfully!


In [13]:
feature_store = pd.read_csv(
    "../data/processed/feature_store.csv"
)

feature_store.head()

,Unnamed: 0,name,main_category,sub_category,image,link,ratings,no_of_ratings,discount_price,actual_price,discount_percentage,price_bucket,category_popularity,subcategory_popularity,popularity_score
0,0,"Redmi 10 Power (Power Black, 8GB RAM, 128GB St...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81eM15lVcJ...,https://www.amazon.in/Redmi-Power-Black-128GB-...,4.0,965.0,10999.0,18999.0,42.107479,Luxury,9600,9600,27.492655
1,1,"OnePlus Nord CE 2 Lite 5G (Blue Tide, 6GB RAM,...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71AvQd3Vzq...,https://www.amazon.in/OnePlus-Nord-Lite-128GB-...,4.3,113956.0,18999.0,19999.0,5.000250,Luxury,9600,9600,50.067379
2,2,OnePlus Bullets Z2 Bluetooth Wireless in Ear E...,"tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/51UhwaQXCp...,https://www.amazon.in/Oneplus-Bluetooth-Wirele...,4.2,90304.0,1999.0,2299.0,13.049152,Medium,9600,9600,47.925982
3,3,"Samsung Galaxy M33 5G (Mystique Green, 6GB, 12...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/81I3w4J6yj...,https://www.amazon.in/Samsung-Mystique-Storage...,4.1,24863.0,15999.0,24999.0,36.001440,Luxury,9600,9600,41.496823
4,4,"OnePlus Nord CE 2 Lite 5G (Black Dusk, 6GB RAM...","tv, audio & cameras",All Electronics,https://m.media-amazon.com/images/I/71V--WZVUI...,https://www.amazon.in/OnePlus-Nord-Black-128GB...,4.3,113956.0,18999.0,19999.0,5.000250,Luxury,9600,9600,50.067379
